# S5 — Season pressing analysis (style + efficiency)

Scores **any (held-out) season** with a trained bundle and derives the full team
pressing analysis. All computation lives in **`src/press_analysis.py`** — the CLI
`apply_season.py` calls the same functions, so notebook and batch runs cannot drift.

**Choosing the season = setting environment variables** (or editing the config cell):

| env var | meaning | default |
|---|---|---|
| `MODEL` | trained bundle (per-season layer from `calibrate_season.py`) | 25/26 bundle |
| `FEAT_PARQUET` | the season's Stage-3 feature parquet | 25/26 file |
| `LABELS_CSV` | the season's Stage-1 labels csv | 25/26 file |
| `EVENTS_DIR` | the season's raw event JSON folder | `$STATSBOMB_DIR/…` |
| `SEASON_LABEL` | free text used in figure titles | `2025/26` |
| `TEAM_ORDER` | optional file fixing fingerprint row order (one team/line) | regain-rate order |
| `OUTDIR` | where tables/figures/lineage.txt are written | `analysis_out/` |

**Lineage check (warn-and-continue).** The bundle records which matches its ranker
was trained on and which fitted its isotonic layer. If the analysed season overlaps,
you get a **warning** (training overlap) or a **note** (calibration overlap) — the
run always continues, and the verdict is persisted to `lineage.txt`. A training-season
analysis belongs in S4's OOF sections instead.


In [ ]:
# 1 — Config: season selection via environment variables
import os, sys, warnings
import numpy as np, pandas as pd
import joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

HPN = os.environ.get("HPN_DIR") or (os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.path.join(HPN, "src"))
from hpn_features import build_feature_matrix, FEATURES
import press_analysis as pa

STATSBOMB_DIR = os.environ.get("STATSBOMB_DIR", os.path.dirname(HPN))
MODEL_PATH   = os.environ.get("MODEL",        os.path.join(HPN, "hpn_xgb_outcome_calibrated_2526.joblib"))
FEAT_PARQUET = os.environ.get("FEAT_PARQUET", os.path.join(HPN, "hpn_carrier_features_2025_2026.parquet"))
LABELS_CSV   = os.environ.get("LABELS_CSV",   os.path.join(HPN, "labels_2025_2026.csv"))
EVENTS_DIR   = os.environ.get("EVENTS_DIR",   os.path.join(STATSBOMB_DIR, "events"))
SEASON_LABEL = os.environ.get("SEASON_LABEL", "2025/26")
TEAM_ORDER   = os.environ.get("TEAM_ORDER")          # optional: file with one team per line
OUTDIR       = os.environ.get("OUTDIR",       os.path.join(HPN, "analysis_out"))
print(f"season={SEASON_LABEL} | model={os.path.basename(MODEL_PATH)} | out={OUTDIR}")


## Load & lineage check

The feature builder emits the full 30-column matrix; the model scores on its own
trained feature subset (`bundle['features']`, the 17 de-collinearised columns —
first-frame temporal diffs stay `NaN`, natively handled by XGBoost).


In [ ]:
# 2 — Features + bundle + lineage
dft = build_feature_matrix(FEAT_PARQUET, LABELS_CSV, EVENTS_DIR)
lab = pd.read_csv(LABELS_CSV); lab["match_id"] = lab["match_id"].astype(str)
print(f"features: {dft.shape} | matches {dft['match_id'].nunique()} | {len(FEATURES)}-col matrix")

bundle = joblib.load(MODEL_PATH)
print(f"model: {os.path.basename(MODEL_PATH)} | {len(bundle['features'])} feats | "
      f"classes {list(bundle['label_encoder'].classes_)} | calib={bundle.get('calib_method')}")
seen_train, seen_calib, lineage_msgs = pa.check_lineage(bundle, dft["match_id"].unique())


## Step valuation — the press-value signal $v_t$

Every anchor gets calibrated probabilities $(p_s, p_f)$; differencing them within a
sequence gives the **step press-value** $v_t = \Delta p_s - \Delta p_f$ (VAEP-style:
how much this step swung the press toward a regain). Each step is also labelled by an
either-axis rule (global $\delta$ = 25th pct of $|\Delta p|$): **Effective** ($p_s\uparrow$,
$p_f\!\downarrow$/flat), **Beaten** ($p_f\uparrow$), **Risky-fav/-adv** (both rise),
**De-escalation**, **Negligible**. First frame of a sequence has no $t\!-\!1$ → no label.


In [ ]:
# 3 — Score + step table (labels distribution + an example sequence)
p_s, p_f = pa.score(dft, bundle)
vp = pa.build_step_table(dft, lab, p_s, p_f)
print(vp.loc[vp["vt"].notna(), "label"].value_counts().to_string())

# example: the season's longest sequence, step by step
_k = vp.groupby("seqkey").size().idxmax()
_s = vp[vp.seqkey == _k].sort_values("ev_pos")
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(_s.ev_pos, _s.p_s, "o-", color="#2ca02c", label="P(success)")
ax[0].plot(_s.ev_pos, _s.p_f, "o-", color="#d62728", label="P(fail)")
ax[0].legend(); ax[0].set_ylabel("calibrated prob")
ax[0].set_title(f"longest sequence {_k} — {_s.press_team.iloc[0]} pressing {_s.pressed_team.iloc[0]}")
ax[1].bar(_s.ev_pos, _s.vt.fillna(0), color=["#2ca02c" if v > 0 else "#d62728" for v in _s.vt.fillna(0)])
ax[1].axhline(0, color="k", lw=.6); ax[1].set_ylabel("$v_t$"); ax[1].set_xlabel("event step")
plt.tight_layout(); plt.show()


## Efficiency analysis — does the press win the ball back?

Four complementary team metrics (per-step mean $v_t$ is deliberately dropped — dividing
by step count penalises teams that sustain longer sequences):

1. **regain_rate** — share of the team's high-press sequences ending in a recovery
   (model-free ground truth);
2. **mean_p_s / mean_p_f** — the calibrated model's average outlook over the team's frames;
3. **vt_per_seq** $= \sum v_t / n_{seq}$ — length-neutral net probability swing per press —
   and **resid_vt**, its length-adjusted residual vs a league per-step baseline;
4. **eff_over_EB** $= E/(E{+}B)$ — share of decisive steps that are Effective rather than
   Beaten (process cleanliness).


In [ ]:
# 4 — Team tables + efficiency figures
intensity, team = pa.team_tables(vp)
display(team.round(4))
fig1 = pa.fig_intensity_efficiency(intensity, team, SEASON_LABEL); plt.show()
fig2 = pa.fig_efficiency(team, SEASON_LABEL); plt.show()


**Reading the quadrant map.** $x$ = pressing intensity $I_T$ (per-match total pressure
on the carrier = volume × per-engagement pressure), $y$ = regain rate; bubble size =
value/seq, colour = process cleanliness $E/(E{+}B)$; dashed lines are league medians.
The two axes are largely orthogonal — pressing harder does not imply recovering more.


## Style analysis — how does each team press?

Ten oriented, z-scored descriptors per team (higher = more aggressive): seven scalar
levels (carrier pressure, swarm, tightness, press height, outlet-denial breadth &
best-pass denial, boundary trap), two rates (press frequency, sequence length) and one
momentum term (Δ carrier pressure). Row order: `TEAM_ORDER` file if provided (e.g. the
final league table), else regain-rate order.


In [ ]:
# 5 — Pressing-style fingerprint
Zt, row_title = pa.fingerprint_matrix(vp, intensity, team, TEAM_ORDER)
fig3 = pa.fig_fingerprint(Zt, row_title, SEASON_LABEL); plt.show()


In [ ]:
# 6 — Drivers: which pressure-channel changes raise v_t?
drivers, r2 = pa.drivers_table(vp)
print(f"joint linear R^2 = {r2:.3f} (channels are collinear — read marginal r alongside)")
display(drivers.round(4))
fig4 = pa.fig_drivers(drivers, SEASON_LABEL); plt.show()


**Reading the drivers.** Bars are the Pearson $r$ of each within-sequence channel Δ
with $v_t$. Correlations are weak by construction (press value is nonlinear and
contextual); the Effective-vs-Beaten column means in the table preserve the intuitive
directions in the decisive tails.


In [ ]:
# 7 — Persist all outputs (4 csv + 4 figures + lineage.txt)
figs = {"fig_intensity_landscape.png": fig1, "fig_efficiency.png": fig2,
        "fig_style_fingerprint.png": fig3, "fig_drivers.png": fig4}
pa.write_outputs(OUTDIR, intensity, team, vp, drivers, figs, lineage_msgs)


---
**Switching seasons.** Run Stages 1+3 on the new season's raw JSON, fit its isotonic
layer with `calibrate_season.py`, then point `MODEL` / `FEAT_PARQUET` / `LABELS_CSV` /
`EVENTS_DIR` / `SEASON_LABEL` here and *Run All*. Zero code changes — and the lineage
check will confirm the season is fully held-out.
